In [ ]:
# !pip install yfinance
# !pip install TA-Lib
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install seaborn

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION — EMA ATR Trend Strategy
# ════════════════════════════════════════════════════════════════

TICKER = 'QQQ'
START_DATE = '2018-01-01'

# ── Core strategy params ──
FAST_EMA = 12
SLOW_EMA = 26
ATR_PERIOD = 14
ATR_TRAIL_MULT = 2.5       # Trailing stop: highest_close - ATR_TRAIL_MULT * ATR
ADX_MIN = 20               # Only enter when ADX >= this (trend strength filter)
PULLBACK_ENABLED = True    # Allow re-entry on pullback to fast EMA in uptrend
PULLBACK_ATR_MULT = 0.5    # Max distance from fast EMA for pullback entry

# ── Backtest settings (standard) ──
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
FREQ = 'D'
TRAIN_RATIO = 0.60

# ── Multi-asset scan tickers ──
SCAN_TICKERS = ['QQQ', 'SPY', 'NVDA', 'AMD', 'META', 'TSLA', 'AAPL', 'MSFT',
                'BTC-USD', 'ETH-USD', 'SOL-USD', 'GC=F']

# Download data
print(f'Downloading {TICKER}...')
stock_data = download(TICKER, START_DATE)
print(f'{TICKER}: {len(stock_data)} bars ({stock_data.index[0].date()} to {stock_data.index[-1].date()})')

# Also download multi-asset for scanning later
print(f'\nDownloading {len(SCAN_TICKERS)} scan tickers...')
all_data = download_multi(SCAN_TICKERS, START_DATE)

In [ ]:
# ════════════════════════════════════════════════════════════════
# TECHNICAL INDICATORS
# ════════════════════════════════════════════════════════════════

close_arr = stock_data['Close'].values.astype(float)
high_arr = stock_data['High'].values.astype(float)
low_arr = stock_data['Low'].values.astype(float)
idx = stock_data.index

indicators_df = pd.DataFrame({
    'Close': close_arr,
    'EMA_Fast': talib.EMA(close_arr, FAST_EMA),
    'EMA_Slow': talib.EMA(close_arr, SLOW_EMA),
    'ATR': talib.ATR(high_arr, low_arr, close_arr, ATR_PERIOD),
    'ADX': talib.ADX(high_arr, low_arr, close_arr, ATR_PERIOD),
    'RSI_14': talib.RSI(close_arr, 14),
}, index=idx)

print('Indicators computed.')
indicators_df.tail(5)

In [ ]:
# ════════════════════════════════════════════════════════════════
# PREPARE PRICE SERIES + IS/OOS SPLIT
# ════════════════════════════════════════════════════════════════

close_s = stock_data['Close'].astype(float).squeeze(); close_s.name = 'price'
high_s = stock_data['High'].astype(float).squeeze()
low_s = stock_data['Low'].astype(float).squeeze()
vol_s = stock_data['Volume'].astype(float).squeeze()

split_idx = int(len(close_s) * TRAIN_RATIO)
split_date = close_s.index[split_idx]

train_close = close_s.iloc[:split_idx].copy()
train_high = high_s.iloc[:split_idx].copy()
train_low = low_s.iloc[:split_idx].copy()
train_vol = vol_s.iloc[:split_idx].copy()

val_close = close_s.iloc[split_idx:].copy()
val_high = high_s.iloc[split_idx:].copy()
val_low = low_s.iloc[split_idx:].copy()
val_vol = vol_s.iloc[split_idx:].copy()

print(f'Full sample: {close_s.index[0].date()} to {close_s.index[-1].date()} ({len(close_s)} bars)')
print(f'In-sample:   {train_close.index[0].date()} to {train_close.index[-1].date()} ({len(train_close)} bars)')
print(f'OOS:         {val_close.index[0].date()} to {val_close.index[-1].date()} ({len(val_close)} bars)')

## EMA ATR Trend Strategy

**Core idea**: EMA crossover for trend direction + ATR trailing stop for dynamic exits.

**Entry signals**:
1. **Crossover entry**: Fast EMA crosses above Slow EMA AND ADX >= threshold (confirms trend strength)
2. **Pullback re-entry**: In uptrend (fast > slow), price pulls back within 0.5 ATR of fast EMA then bounces

**Exit signals**:
1. **ATR trailing stop**: Price drops below `highest_since_entry - ATR_mult * ATR` (adaptive to volatility)
2. **EMA death cross**: Fast EMA crosses below Slow EMA (trend reversal)

**Why ATR trailing stops beat fixed stops**:
- In low-vol periods: tight stop protects gains
- In high-vol periods: wide stop avoids getting shaken out
- Positive skew: lets winners run, cuts losers based on actual market conditions

**Anti-lookahead**: All signals use previous bar's indicators. Entry/exit on current bar uses only information available at prior close.

In [ ]:
# ════════════════════════════════════════════════════════════════
# SIGNAL ENGINE
# ════════════════════════════════════════════════════════════════

def generate_signals(close, high, low, volume,
                     fast_ema=12, slow_ema=26, atr_period=14,
                     trail_atr_mult=2.5, adx_min=20,
                     pullback=True, pullback_atr_mult=0.5):
    """
    EMA ATR signal engine with trailing stop.
    
    Uses a bar-by-bar loop to properly track trailing stop state.
    All signals use PREVIOUS bar's indicator values (anti-lookahead).
    
    Returns: entries Series (bool), exits Series (bool), trade_log list
    """
    c = close.values.astype(float)
    h = high.values.astype(float)
    l = low.values.astype(float)
    idx = close.index
    n = len(c)
    
    # Compute indicators
    fast_vals = talib.EMA(c, timeperiod=fast_ema)
    slow_vals = talib.EMA(c, timeperiod=slow_ema)
    atr_vals = talib.ATR(h, l, c, timeperiod=atr_period)
    adx_vals = talib.ADX(h, l, c, timeperiod=atr_period)
    
    entries = np.zeros(n, dtype=bool)
    exits = np.zeros(n, dtype=bool)
    
    in_trade = False
    highest = 0.0
    stop = 0.0
    trade_log = []  # track entry/exit reasons
    
    min_bar = max(slow_ema, atr_period) + 2
    
    for i in range(min_bar, n):
        if np.isnan(fast_vals[i]) or np.isnan(slow_vals[i]) or np.isnan(atr_vals[i]):
            continue
        
        # Previous bar values (anti-lookahead)
        pf = fast_vals[i - 1]   # prev fast EMA
        ps = slow_vals[i - 1]   # prev slow EMA
        pa = atr_vals[i - 1]    # prev ATR
        padx = adx_vals[i - 1] if not np.isnan(adx_vals[i - 1]) else 0
        pc = c[i - 1]           # prev close
        cc = c[i]               # current close
        
        # 2-bars-ago for crossover detection
        p2f = fast_vals[i - 2] if i >= 2 else np.nan
        p2s = slow_vals[i - 2] if i >= 2 else np.nan
        
        if in_trade:
            # Update trailing stop
            if cc > highest:
                highest = cc
                stop = highest - trail_atr_mult * pa
            
            # EXIT 1: ATR trailing stop hit
            if cc < stop:
                exits[i] = True
                in_trade = False
                trade_log.append(('exit_atr_stop', idx[i]))
                continue
            
            # EXIT 2: EMA death cross
            if not np.isnan(p2f) and p2f >= p2s and pf < ps:
                exits[i] = True
                in_trade = False
                trade_log.append(('exit_ema_cross', idx[i]))
                continue
        else:
            # ENTRY 1: EMA golden cross + ADX filter
            if not np.isnan(p2f) and p2f <= p2s and pf > ps and padx >= adx_min:
                entries[i] = True
                in_trade = True
                highest = cc
                stop = cc - trail_atr_mult * pa
                trade_log.append(('entry_crossover', idx[i]))
                continue
            
            # ENTRY 2: Pullback re-entry in uptrend
            if pullback and pf > ps and padx >= adx_min:
                dist = abs(pc - pf)
                if dist <= pullback_atr_mult * pa and cc > pc:
                    entries[i] = True
                    in_trade = True
                    highest = cc
                    stop = cc - trail_atr_mult * pa
                    trade_log.append(('entry_pullback', idx[i]))
                    continue
    
    return (pd.Series(entries, index=idx, dtype=bool),
            pd.Series(exits, index=idx, dtype=bool),
            trade_log)


def run_backtest(close, entries, exits, init_cash=100_000, fees=0.0005, slippage=0.0005):
    """Run vectorbt backtest and extract metrics."""
    pf = vbt.Portfolio.from_signals(
        close=close, entries=entries, exits=exits,
        init_cash=init_cash, fees=fees, slippage=slippage, freq='D'
    )
    
    trades = pf.trades
    n_trades = len(trades)
    years = max((close.index[-1] - close.index[0]).days / 365.25, 0.01)
    
    # Trade-level stats
    win_rate = np.nan
    profit_factor = np.nan
    expectancy = 0.0
    avg_win = 0.0
    avg_loss = 0.0
    largest_win = 0.0
    largest_loss = 0.0
    
    if n_trades > 0:
        tr = np.asarray(
            trades.returns.values if hasattr(trades.returns, 'values') else trades.returns
        ).ravel()
        if tr.size > 0:
            pos = tr[tr > 0]
            neg = tr[tr < 0]
            win_rate = len(pos) / len(tr) * 100 if len(tr) else np.nan
            profit_factor = pos.sum() / abs(neg.sum()) if len(neg) and abs(neg.sum()) > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win = float(pos.mean()) if len(pos) else 0
            avg_loss = float(abs(neg.mean())) if len(neg) else 0
            largest_win = float(pos.max()) if len(pos) else 0
            largest_loss = float(neg.min()) if len(neg) else 0
    
    return {
        'total_return': float(pf.total_return()),
        'ann_return': float(pf.annualized_return(freq='D')),
        'sharpe': float(pf.sharpe_ratio(freq='D')),
        'sortino': float(pf.sortino_ratio(freq='D')),
        'max_dd': float(pf.max_drawdown()),
        'ann_vol': float(pf.annualized_volatility(freq='D')),
        'calmar': float(pf.annualized_return(freq='D')) / abs(float(pf.max_drawdown()))
                  if abs(float(pf.max_drawdown())) > 1e-9 else 0,
        'n_trades': n_trades,
        'trades_per_year': n_trades / years,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'expectancy': expectancy,
        'avg_win': avg_win,
        'avg_loss': avg_loss,
        'largest_win': largest_win,
        'largest_loss': largest_loss,
        'pf': pf,
    }

print('Signal engine defined.')

In [ ]:
# STRATEGY PARAMETERS

print(f'EMA ATR Trend Strategy — {TICKER}')
print('=' * 55)
print(f'  ── Signal ──')
print(f'  Fast EMA:          {FAST_EMA}')
print(f'  Slow EMA:          {SLOW_EMA}')
print(f'  ATR Period:        {ATR_PERIOD}')
print(f'  ATR Trail Mult:    {ATR_TRAIL_MULT}x')
print(f'  ADX Min:           {ADX_MIN}')
print(f'  Pullback:          {PULLBACK_ENABLED} (within {PULLBACK_ATR_MULT} ATR)')
print(f'')
print(f'  ── Backtest ──')
print(f'  Init Cash:         ${INIT_CASH:,.0f}')
print(f'  Fees:              {FEES*100:.2f}%')
print(f'  Slippage:          {SLIPPAGE*100:.2f}%')
print('=' * 55)

In [ ]:
# ════════════════════════════════════════════════════════════════
# IS / OOS CONSISTENCY CHECK
# ════════════════════════════════════════════════════════════════

_sig_params = dict(
    fast_ema=FAST_EMA, slow_ema=SLOW_EMA, atr_period=ATR_PERIOD,
    trail_atr_mult=ATR_TRAIL_MULT, adx_min=ADX_MIN,
    pullback=PULLBACK_ENABLED, pullback_atr_mult=PULLBACK_ATR_MULT,
)

# IS
e_is, x_is, log_is = generate_signals(train_close, train_high, train_low, train_vol, **_sig_params)
m_is = run_backtest(train_close, e_is, x_is, INIT_CASH, FEES, SLIPPAGE)

# OOS
e_oos, x_oos, log_oos = generate_signals(val_close, val_high, val_low, val_vol, **_sig_params)
m_oos = run_backtest(val_close, e_oos, x_oos, INIT_CASH, FEES, SLIPPAGE)

# Entry type breakdown
is_cross = sum(1 for t, _ in log_is if t == 'entry_crossover')
is_pull = sum(1 for t, _ in log_is if t == 'entry_pullback')
is_exit_atr = sum(1 for t, _ in log_is if t == 'exit_atr_stop')
is_exit_ema = sum(1 for t, _ in log_is if t == 'exit_ema_cross')

oos_cross = sum(1 for t, _ in log_oos if t == 'entry_crossover')
oos_pull = sum(1 for t, _ in log_oos if t == 'entry_pullback')
oos_exit_atr = sum(1 for t, _ in log_oos if t == 'exit_atr_stop')
oos_exit_ema = sum(1 for t, _ in log_oos if t == 'exit_ema_cross')

print('=' * 85)
print(f'IS vs OOS CONSISTENCY CHECK — {TICKER}')
print('=' * 85)
print(f'{"Metric":<24} {"IS":>12} {"OOS":>12} {"Diff":>12}')
print('-' * 60)
for key, fmt in [('total_return', '.2%'), ('ann_return', '.2%'), ('sharpe', '.3f'),
                  ('sortino', '.3f'), ('max_dd', '.2%'), ('calmar', '.2f'),
                  ('ann_vol', '.2%'), ('win_rate', '.1f'), ('n_trades', '.0f'),
                  ('trades_per_year', '.1f'), ('profit_factor', '.2f'),
                  ('expectancy', '.4f')]:
    v_is = m_is.get(key, 0) or 0
    v_oos = m_oos.get(key, 0) or 0
    diff = v_oos - v_is
    suffix = '%' if key == 'win_rate' else ''
    print(f'{key:<24} {v_is:>12{fmt}}{suffix} {v_oos:>12{fmt}}{suffix} {diff:>+12{fmt}}{suffix}')
print('=' * 85)

print(f'\nEntry types   IS: {is_cross} crossover + {is_pull} pullback | OOS: {oos_cross} crossover + {oos_pull} pullback')
print(f'Exit types    IS: {is_exit_atr} ATR stop + {is_exit_ema} EMA cross | OOS: {oos_exit_atr} ATR stop + {oos_exit_ema} EMA cross')

sharpe_diff = abs(m_is['sharpe'] - m_oos['sharpe'])
if m_oos['sharpe'] > 0.5 and sharpe_diff < 0.5:
    print('\n✅ CONSISTENT — OOS Sharpe healthy, IS/OOS gap small')
elif m_oos['sharpe'] > 0 and sharpe_diff < 1.0:
    print('\n⚠️ ACCEPTABLE — OOS positive but some decay')
else:
    print('\n❌ INCONSISTENT — significant IS/OOS divergence')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# FULL-SAMPLE RUN + MULTI-ASSET SCAN
# ════════════════════════════════════════════════════════════════════════

# Full sample on primary ticker
e_full, x_full, log_full = generate_signals(close_s, high_s, low_s, vol_s, **_sig_params)
m_full = run_backtest(close_s, e_full, x_full, INIT_CASH, FEES, SLIPPAGE)
pf_full = m_full['pf']

print('=' * 90)
print(f'EMA ATR TREND — {TICKER} | EMA({FAST_EMA},{SLOW_EMA}) ATR({ATR_PERIOD})x{ATR_TRAIL_MULT} ADX>={ADX_MIN}')
print(f'Period: {close_s.index[0].date()} to {close_s.index[-1].date()}')
print('=' * 90)
print(f'{"Metric":<24} {"Full":>12} {"IS":>12} {"OOS":>12}')
print('-' * 60)
for key, fmt in [('total_return', '.2%'), ('ann_return', '.2%'), ('sharpe', '.3f'),
                  ('sortino', '.3f'), ('max_dd', '.2%'), ('calmar', '.2f'),
                  ('ann_vol', '.2%'), ('win_rate', '.1f'), ('n_trades', '.0f'),
                  ('trades_per_year', '.1f'), ('profit_factor', '.2f')]:
    vals = [m_full.get(key, 0) or 0, m_is.get(key, 0) or 0, m_oos.get(key, 0) or 0]
    suffix = '%' if key == 'win_rate' else ''
    print(f'{key:<24} {vals[0]:>12{fmt}}{suffix} {vals[1]:>12{fmt}}{suffix} {vals[2]:>12{fmt}}{suffix}')
print('=' * 90)

# Multi-asset scan
print(f'\n── MULTI-ASSET SCAN ({len(all_data)} tickers) ──')
print(f'{"Ticker":<10} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Trades":>8} {"T/Yr":>6} {"WinRate":>8} {"PF":>6}')
print('-' * 72)
multi_results = []
for tk, df in all_data.items():
    try:
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]
        tc = df['Close'].astype(float).squeeze()
        th = df['High'].astype(float).squeeze()
        tl = df['Low'].astype(float).squeeze()
        tv = df['Volume'].astype(float).squeeze()
        # OOS only
        sp = int(len(tc) * TRAIN_RATIO)
        e, x, _ = generate_signals(tc.iloc[sp:], th.iloc[sp:], tl.iloc[sp:], tv.iloc[sp:], **_sig_params)
        m = run_backtest(tc.iloc[sp:], e, x, INIT_CASH, FEES, SLIPPAGE)
        multi_results.append({'ticker': tk, **{k: v for k, v in m.items() if k != 'pf'}})
        wr = m['win_rate'] if not np.isnan(m['win_rate']) else 0
        pf_val = m['profit_factor'] if not np.isnan(m['profit_factor']) and m['profit_factor'] < 100 else 0
        print(f'{tk:<10} {m["sharpe"]:>8.3f} {m["total_return"]:>10.2%} {m["max_dd"]:>10.2%} '
              f'{m["n_trades"]:>8d} {m["trades_per_year"]:>6.1f} {wr:>7.1f}% {pf_val:>6.2f}')
    except Exception as ex:
        print(f'{tk:<10} ERROR: {ex}')

if multi_results:
    mdf = pd.DataFrame(multi_results)
    avg_sr = mdf['sharpe'].mean()
    pos_sr = (mdf['sharpe'] > 0).sum()
    print(f'\nAvg Sharpe: {avg_sr:.3f} | Positive: {pos_sr}/{len(mdf)}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PERFORMANCE DASHBOARD
# ════════════════════════════════════════════════════════════════════════

# Compute derived data
returns = pf_full.returns()
cum_returns = (1 + returns).cumprod() - 1
equity = pf_full.value()
running_max = equity.cummax()
drawdown = (equity - running_max) / running_max * 100

monthly_rets = returns.resample('M').apply(lambda x: (1 + x).prod() - 1) * 100
monthly_data = pd.DataFrame({'year': monthly_rets.index.year, 'month': monthly_rets.index.month, 'return': monthly_rets.values})
pivot = monthly_data.pivot_table(values='return', index='month', columns='year')

yearly_rets = returns.resample('YE').apply(lambda x: (1 + x).prod() - 1) * 100

rolling_sharpe = returns.rolling(252).mean() / returns.rolling(252).std() * np.sqrt(252)
rolling_vol = returns.rolling(252).std() * np.sqrt(252) * 100

# Buy & hold benchmark
bh_ret = close_s.pct_change().fillna(0)
bh_cum = (1 + bh_ret).cumprod() - 1

# ATR trailing stop visualization data
atr_full = pd.Series(talib.ATR(high_s.values, low_s.values, close_s.values, ATR_PERIOD), index=close_s.index)

# Trade PnL
trades_obj = pf_full.trades
trade_pnl = np.asarray(
    trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else trades_obj.pnl
).ravel() if len(trades_obj) > 0 else np.array([])
trade_rets = np.asarray(
    trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else trades_obj.returns
).ravel() if len(trades_obj) > 0 else np.array([])

# ── Dashboard ──
fig = plt.figure(figsize=(22, 28))
fig.suptitle(f'EMA ATR Trend Strategy — {TICKER} | EMA({FAST_EMA},{SLOW_EMA}) ATR x{ATR_TRAIL_MULT} ADX>={ADX_MIN}',
             fontsize=20, fontweight='bold', y=0.995)

gs = gridspec.GridSpec(5, 6, figure=fig, hspace=0.42, wspace=0.45,
                       top=0.97, bottom=0.03, left=0.06, right=0.97)

# ── Row 1: Equity Curve + Key Metrics ──
ax1 = fig.add_subplot(gs[0, :5])
ax1.fill_between(cum_returns.index, cum_returns.values * 100, alpha=0.25, color='#2ca02c')
ax1.plot(cum_returns.index, cum_returns.values * 100, color='#2ca02c', linewidth=1.8)
# Mark entries and exits
entry_dates = close_s.index[e_full]
exit_dates = close_s.index[x_full]
for ed in entry_dates:
    if ed in cum_returns.index:
        ax1.axvline(ed, color='green', alpha=0.08, linewidth=0.5)
for xd in exit_dates:
    if xd in cum_returns.index:
        ax1.axvline(xd, color='red', alpha=0.08, linewidth=0.5)
ax1.axvline(x=split_date, color='red', linestyle=':', alpha=0.5, label='IS/OOS Split')
ax1.set_title('Cumulative Returns (Net)', fontsize=15, fontweight='bold')
ax1.set_ylabel('Return (%)', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}%'))

metrics_text = (
    f'KEY METRICS\n'
    f'Total Return: {m_full["total_return"]*100:>8.1f}%\n'
    f'CAGR:         {m_full["ann_return"]*100:>8.1f}%\n'
    f'Sharpe:       {m_full["sharpe"]:>8.2f}\n'
    f'Sortino:      {m_full["sortino"]:>8.2f}\n'
    f'Max DD:       {m_full["max_dd"]*100:>7.1f}%\n'
    f'Win Rate:     {m_full["win_rate"]:>7.1f}%\n'
    f'Trades:       {m_full["n_trades"]:>7d}\n'
    f'Profit F:     {m_full["profit_factor"]:>7.2f}'
)
ax_m = fig.add_subplot(gs[0, 5])
ax_m.axis('off')
ax_m.text(0.05, 0.95, metrics_text, transform=ax_m.transAxes,
          fontsize=11, verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='#333', alpha=0.95))

# ── Row 2: Drawdown + Monthly Heatmap ──
ax2 = fig.add_subplot(gs[1, :4])
ax2.fill_between(drawdown.index, drawdown.values, alpha=0.45, color='#8b0000')
ax2.plot(drawdown.index, drawdown.values, color='#8b0000', linewidth=0.8)
worst_idx = drawdown.idxmin()
ax2.scatter([worst_idx], [drawdown.min()], color='red', s=100, zorder=5, edgecolors='black')
ax2.set_title('Drawdown Over Time', fontsize=15, fontweight='bold')
ax2.set_ylabel('Drawdown (%)', fontsize=12)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 4:])
month_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
y_labels = [month_labels[m-1] for m in pivot.index]
sns.heatmap(pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f',
            ax=ax3, cbar_kws={'label': 'Return (%)', 'shrink': 0.8},
            yticklabels=y_labels, linewidths=0.5, linecolor='white')
ax3.set_title('Monthly Returns (%)', fontsize=15, fontweight='bold')

# ── Row 3: Trade Analysis (2x2) ──
ax4 = fig.add_subplot(gs[2, :3])
if len(trade_pnl) > 0:
    colors_pnl = ['#2ca02c' if p > 0 else '#d62728' for p in trade_pnl]
    ax4.bar(range(len(trade_pnl)), trade_pnl, color=colors_pnl, alpha=0.7, width=1.0)
    ax4.axhline(0, color='black', linewidth=0.5)
    ax4.axhline(np.mean(trade_pnl), color='blue', linestyle='--', linewidth=1.5,
                label=f'Mean: ${np.mean(trade_pnl):,.0f}')
    ax4.legend(fontsize=10)
ax4.set_title('Trade P&L Sequence', fontsize=15, fontweight='bold')
ax4.set_xlabel('Trade #', fontsize=12)
ax4.set_ylabel('P&L ($)', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')

ax5 = fig.add_subplot(gs[2, 3:])
if len(trade_rets) > 0:
    ax5.hist(trade_rets * 100, bins=min(40, max(10, len(trade_rets)//3)),
             color='steelblue', alpha=0.7, edgecolor='white')
    ax5.axvline(np.mean(trade_rets) * 100, color='blue', linestyle='--', linewidth=2,
                label=f'Mean: {np.mean(trade_rets)*100:.2f}%')
    ax5.axvline(0, color='black', linewidth=0.5)
    ax5.legend(fontsize=10)
ax5.set_title('Trade Return Distribution', fontsize=15, fontweight='bold')
ax5.set_xlabel('Return (%)', fontsize=12)
ax5.set_ylabel('Count', fontsize=12)
ax5.grid(True, alpha=0.3)

# ── Row 4: Rolling Sharpe + Rolling Vol + Yearly ──
ax6 = fig.add_subplot(gs[3, :2])
ax6.plot(rolling_sharpe.index, rolling_sharpe.values, color='purple', linewidth=1)
mean_rs = rolling_sharpe.dropna().mean()
ax6.axhline(y=mean_rs, color='blue', linestyle='--', linewidth=1.8,
            label=f'Mean: {mean_rs:.2f}')
ax6.axhline(y=0, color='black', linewidth=0.5, alpha=0.3)
ax6.axvline(x=split_date, color='red', linestyle=':', alpha=0.5)
ax6.set_title('Rolling Sharpe (252 days)', fontsize=15, fontweight='bold')
ax6.set_ylabel('Sharpe Ratio', fontsize=12)
ax6.legend(fontsize=11)
ax6.grid(True, alpha=0.3)

ax7 = fig.add_subplot(gs[3, 2:4])
ax7.plot(rolling_vol.index, rolling_vol.values, color='#ff8c00', linewidth=1)
ax7.axvline(x=split_date, color='red', linestyle=':', alpha=0.5)
ax7.set_title('Rolling Volatility (252 days)', fontsize=15, fontweight='bold')
ax7.set_ylabel('Annualized Vol (%)', fontsize=12)
ax7.grid(True, alpha=0.3)

ax8 = fig.add_subplot(gs[3, 4:])
years_arr = yearly_rets.index.year
colors_yr = ['#2ca02c' if v >= 0 else '#d62728' for v in yearly_rets.values]
ax8.bar(years_arr, yearly_rets.values, color=colors_yr, alpha=0.85, edgecolor='darkgreen', linewidth=0.5)
if len(yearly_rets) > 1:
    complete_years = yearly_rets.iloc[:-1]
    mean_ret = complete_years.mean()
    ax8.axhline(y=mean_ret, color='blue', linestyle='--', linewidth=1.8,
                label=f'Mean: {mean_ret:.1f}%')
    ax8.legend(fontsize=10)
ax8.set_title('Yearly Results', fontsize=15, fontweight='bold')
ax8.set_ylabel('Return (%)', fontsize=12)
ax8.grid(True, alpha=0.3, axis='y')

# ── Row 5: Strategy vs B&H + Multi-Asset OOS comparison ──
ax9 = fig.add_subplot(gs[4, :3])
ax9.plot(cum_returns.index, cum_returns.values * 100, color='#2ca02c',
         linewidth=1.8, label='Strategy')
ax9.plot(bh_cum.index, bh_cum.values * 100, color='gray',
         linewidth=1.5, alpha=0.7, label=f'Buy & Hold {TICKER}', linestyle='--')
ax9.axvline(x=split_date, color='red', linestyle=':', alpha=0.5, label='IS/OOS Split')
ax9.set_title('Strategy vs Buy & Hold', fontsize=15, fontweight='bold')
ax9.set_ylabel('Return (%)', fontsize=12)
ax9.legend(fontsize=10)
ax9.grid(True, alpha=0.3)

ax10 = fig.add_subplot(gs[4, 3:])
if multi_results:
    mdf_sorted = pd.DataFrame(multi_results).sort_values('sharpe', ascending=True)
    colors_multi = ['#2ca02c' if s > 0 else '#d62728' for s in mdf_sorted['sharpe']]
    ax10.barh(mdf_sorted['ticker'], mdf_sorted['sharpe'], color=colors_multi, alpha=0.85)
    ax10.axvline(0, color='black', linewidth=0.5)
    for i, (_, row) in enumerate(mdf_sorted.iterrows()):
        ax10.text(row['sharpe'] + 0.02, i, f'{row["sharpe"]:.2f}', va='center', fontsize=8)
ax10.set_title('Multi-Asset OOS Sharpe', fontsize=15, fontweight='bold')
ax10.set_xlabel('Sharpe Ratio', fontsize=12)
ax10.grid(True, alpha=0.2, axis='x')

plt.show()

dashboard_fig = fig
print(f'\nDashboard generated for {TICKER} EMA({FAST_EMA},{SLOW_EMA}) ATR x{ATR_TRAIL_MULT}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# MONTE CARLO SIMULATION — FTMO CHALLENGE
# ════════════════════════════════════════════════════════════════

print('=' * 70)
print('\U0001f3b2 MONTE CARLO SIMULATION — FTMO CHALLENGE FEASIBILITY')
print('=' * 70)

daily_rets = returns.values.ravel()
daily_rets = daily_rets[~np.isnan(daily_rets)]

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
TRADING_DAYS = 30
N_SIMULATIONS = 10_000

print(f'\nStrategy: {TICKER} EMA({FAST_EMA},{SLOW_EMA}) ATR x{ATR_TRAIL_MULT}')
print(f'Observed daily returns: {len(daily_rets)} days')
print(f'  Mean daily return:    {np.mean(daily_rets)*100:.4f}%')
print(f'  Std daily return:     {np.std(daily_rets)*100:.4f}%')

np.random.seed(42)
equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
equity_paths[:, 0] = FTMO_ACCOUNT

passed = np.zeros(N_SIMULATIONS, dtype=bool)
blown = np.zeros(N_SIMULATIONS, dtype=bool)
daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
final_equity = np.zeros(N_SIMULATIONS)

for sim in range(N_SIMULATIONS):
    sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
    eq = FTMO_ACCOUNT
    for day in range(TRADING_DAYS):
        day_start = eq
        eq = eq * (1 + sim_rets[day])
        equity_paths[sim, day + 1] = eq
        if (eq - day_start) / FTMO_ACCOUNT < -MAX_DAILY_LOSS:
            daily_blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT < -MAX_TOTAL_LOSS:
            blown[sim] = True; equity_paths[sim, day+2:] = eq; break
        if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT >= PROFIT_TARGET:
            passed[sim] = True; equity_paths[sim, day+2:] = eq; break
    final_equity[sim] = equity_paths[sim, -1]

n_passed = passed.sum()
n_blown_total = blown.sum()
n_blown_daily = daily_blown.sum()
n_survived = N_SIMULATIONS - n_blown_total - n_blown_daily - n_passed
pass_rate = n_passed / N_SIMULATIONS * 100

print(f'\n{"=" * 70}')
print(f'\U0001f3af MONTE CARLO RESULTS ({N_SIMULATIONS:,} simulations)')
print(f'{"=" * 70}')
print(f'  \u2705 PASSED:              {n_passed:>6,} ({n_passed/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (total DD):    {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)')
print(f'  \u274c BLOWN (daily DD):    {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)')
print(f'  \u23f3 Still trading:       {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)')
print(f'  Median final equity:   ${np.median(final_equity):>10,.0f}')

verdict = '\U0001f7e2 FAVORABLE' if pass_rate >= 50 else '\U0001f7e1 POSSIBLE' if pass_rate >= 25 else '\U0001f7e0 CHALLENGING' if pass_rate >= 10 else '\U0001f534 UNLIKELY'
print(f'\n  FTMO VERDICT: {verdict} — {pass_rate:.1f}% pass rate')
print(f'{"=" * 70}')

# Monte Carlo plot
fig_mc, axes = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [2, 1]})
fig_mc.suptitle(f'FTMO Challenge Simulation | {TICKER} EMA ATR | {N_SIMULATIONS:,} Paths',
                fontsize=16, fontweight='bold')

sample_idx = np.random.choice(N_SIMULATIONS, min(300, N_SIMULATIONS), replace=False)
for i in sample_idx:
    c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
    a = 0.3 if (passed[i] or blown[i] or daily_blown[i]) else 0.08
    axes[0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)
axes[0].axhline(FTMO_ACCOUNT * 1.10, color='green', linestyle='--', linewidth=2.5, label='10% Target ($110K)')
axes[0].axhline(FTMO_ACCOUNT * 0.90, color='red', linestyle='--', linewidth=2.5, label='10% Max Loss ($90K)')
axes[0].set_ylabel('Equity ($)'); axes[0].legend(fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].grid(True, alpha=0.2)

bins = np.linspace(final_equity.min(), final_equity.max(), 60)
axes[1].hist(final_equity, bins=bins, color='#1f77b4', alpha=0.7, edgecolor='white')
axes[1].axvline(np.median(final_equity), color='orange', linestyle='--', linewidth=2,
                label=f'Median: ${np.median(final_equity):,.0f}')
axes[1].axvline(FTMO_ACCOUNT * 1.10, color='green', linestyle='--', linewidth=2)
axes[1].axvline(FTMO_ACCOUNT * 0.90, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Final Equity ($)'); axes[1].set_ylabel('Frequency')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════
# EXPORT — JSON + CSV + PDF
# ════════════════════════════════════════════════════════════════

import json, datetime

STRATEGY_NAME = 'EMA_ATR_Trend'

EXPORT_DIR = 'strategy_exports'
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = '/content/drive/MyDrive/strategy_exports'
except:
    pass

out_dir = os.path.join(EXPORT_DIR, STRATEGY_NAME, TICKER)
os.makedirs(os.path.join(out_dir, 'latest'), exist_ok=True)
os.makedirs(os.path.join(out_dir, 'archive'), exist_ok=True)

run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

summary = {
    'metadata': {
        'strategy': STRATEGY_NAME,
        'ticker': TICKER,
        'run_id': run_id,
        'start_date': str(close_s.index[0].date()),
        'end_date': str(close_s.index[-1].date()),
        'n_bars': len(close_s),
        'freq': FREQ,
    },
    'params': {
        'fast_ema': FAST_EMA, 'slow_ema': SLOW_EMA,
        'atr_period': ATR_PERIOD, 'atr_trail_mult': ATR_TRAIL_MULT,
        'adx_min': ADX_MIN, 'pullback': PULLBACK_ENABLED,
        'pullback_atr_mult': PULLBACK_ATR_MULT,
    },
    'metrics': {
        'full': {k: v for k, v in m_full.items() if k != 'pf'},
        'is': {k: v for k, v in m_is.items() if k != 'pf'},
        'oos': {k: v for k, v in m_oos.items() if k != 'pf'},
    },
    'multi_asset': multi_results if multi_results else [],
    'ftmo_monte_carlo': {
        'pass_rate': pass_rate, 'n_passed': int(n_passed),
        'n_blown_total': int(n_blown_total), 'n_blown_daily': int(n_blown_daily),
        'n_survived': int(n_survived),
    },
}

with open(os.path.join(out_dir, 'latest', 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

returns.to_csv(os.path.join(out_dir, 'latest', 'daily_returns.csv'))

# Save dashboard as PDF
try:
    dashboard_fig.savefig(os.path.join(out_dir, 'latest', 'dashboard.pdf'),
                          dpi=150, bbox_inches='tight', facecolor='white')
    print(f'Dashboard PDF saved.')
except:
    print('Dashboard PDF save failed.')

print(f'\nExported to: {out_dir}/latest/')
print(f'Run ID: {run_id}')
print(f'Files: summary.json, daily_returns.csv, dashboard.pdf')